# ACFT Kaggle 02 - Train Smoke and Resume

Attaches generated chunks, adds a capped public-ASR mix, runs existing Stage 17 with conservative defaults, and versions checkpoints/logs as a private Kaggle Dataset.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

OWNER = os.environ.get("KAGGLE_OWNER", "drsriharshaguthik")
PROFILE = os.environ.get("PROFILE", "smoke")
RUN_TAG = os.environ.get("RUN_TAG", datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S"))
REPO_ROOT = Path(os.environ.get("ACFT_REPO_ROOT", "/kaggle/working/whisper-acft"))
RUN_ROOT = Path(os.environ.get("ACFT_TRAIN_RUN_ROOT", "/kaggle/working/acft_train_runs")) / RUN_TAG
STATE_DIR = RUN_ROOT / "state"
CHECKPOINT_DIR = RUN_ROOT / "checkpoints"
PUBLIC_RATIO = float(os.environ.get("PUBLIC_RATIO", "0.30"))
ENABLE_PUBLIC_ASR = os.environ.get("ENABLE_PUBLIC_ASR", "1") == "1"
ENABLE_COMMON_VOICE = os.environ.get("ENABLE_COMMON_VOICE", "0") == "1"
DRY_RUN_STAGE17 = os.environ.get("DRY_RUN_STAGE17", "0") == "1"
DRY_RUN_PUBLISH = os.environ.get("DRY_RUN_PUBLISH", "0") == "1"
START_FRESH = int(os.environ.get("START_FRESH", "0"))
N_SAMPLES_PER_EPOCH = int(os.environ.get("N_SAMPLES_PER_EPOCH", "32" if PROFILE == "smoke" else "1000"))
PRIVATE_MAX_ROWS = int(os.environ.get("PRIVATE_MAX_ROWS", "64" if PROFILE == "smoke" else "0"))
PUBLIC_MAX_ROWS = int(os.environ.get("PUBLIC_MAX_ROWS", "32" if PROFILE == "smoke" else "5000"))

RUN_ROOT.mkdir(parents=True, exist_ok=True)
STATE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print({"profile": PROFILE, "run_tag": RUN_TAG, "run_root": str(RUN_ROOT)})

In [ ]:
if not (REPO_ROOT / "tools" / "kaggle_acft_helpers.py").exists():
    git_url = os.environ.get("ACFT_GIT_URL", "")
    if git_url:
        subprocess.run(["git", "clone", git_url, str(REPO_ROOT)], check=True)
    else:
        raise FileNotFoundError(
            f"Repo not found at {REPO_ROOT}. Attach repo files or set ACFT_GIT_URL."
        )

sys.path.insert(0, str(REPO_ROOT))
from tools import kaggle_acft_helpers as kh

print("helper", kh.__file__)

In [ ]:
def find_chunks_dir() -> Path:
    explicit = os.environ.get("ACFT_CHUNKS_DIR", "")
    if explicit:
        p = Path(explicit)
        if p.exists():
            return p
    candidates = []
    for root in Path("/kaggle/input").glob("*"):
        for rel in [Path("Record_chunks"), Path("acft_data") / "Record_chunks"]:
            p = root / rel
            if (p / "pairs_manifest_stage15_train_randomized.jsonl").exists() or (p / "pairs_manifest.jsonl").exists():
                candidates.append(p)
    if not candidates:
        raise FileNotFoundError("Attach notebook 01 chunks dataset or set ACFT_CHUNKS_DIR.")
    candidates.sort(key=lambda p: str(p))
    return candidates[-1]


CHUNKS_DIR = find_chunks_dir()
TRAIN_SOURCE = CHUNKS_DIR / "pairs_manifest_stage15_train_randomized.jsonl"
if not TRAIN_SOURCE.exists():
    TRAIN_SOURCE = CHUNKS_DIR / "pairs_manifest_stage13_train.jsonl"
if not TRAIN_SOURCE.exists():
    TRAIN_SOURCE = CHUNKS_DIR / "pairs_manifest.jsonl"
TEST_SOURCE = CHUNKS_DIR / "pairs_manifest_stage13_test.jsonl"

private_rows = kh.read_jsonl(TRAIN_SOURCE)
if PRIVATE_MAX_ROWS > 0:
    private_rows = private_rows[:PRIVATE_MAX_ROWS]
private_train_manifest = RUN_ROOT / "private_train_manifest.jsonl"
kh.write_jsonl(private_train_manifest, private_rows)

private_eval_rows = kh.read_jsonl(TEST_SOURCE) if TEST_SOURCE.exists() else []
private_eval_rows = [dict(row, dataset_scope="private_acft", exclude_from_private_eval=False) for row in private_eval_rows]
private_eval_manifest = RUN_ROOT / "private_eval_manifest.jsonl"
kh.write_jsonl(private_eval_manifest, private_eval_rows)

print({"chunks_dir": str(CHUNKS_DIR), "private_train_rows": len(private_rows), "private_eval_rows": len(private_eval_rows)})

In [ ]:
public_specs = [
    {"name": "librispeech_asr", "config": "clean", "split": "train.100", "text_column": "text", "source": "librispeech-clean-100"},
    {"name": "google/fleurs", "config": "en_us", "split": "train", "text_column": "transcription", "source": "fleurs-en-us"},
]
if ENABLE_COMMON_VOICE:
    public_specs.append({"name": "mozilla-foundation/common_voice_17_0", "config": "en", "split": "train", "text_column": "sentence", "source": "common-voice-en"})

public_rows = []
if ENABLE_PUBLIC_ASR and PUBLIC_MAX_ROWS > 0:
    try:
        public_rows = kh.build_public_asr_manifest_from_hf(
            output_dir=RUN_ROOT / "public_asr_audio",
            specs=public_specs,
            max_rows=PUBLIC_MAX_ROWS,
            seed=17,
        )
    except Exception as exc:
        print("public ASR fetch failed; continuing private-only", repr(exc))
        public_rows = []

mixed_rows = kh.mix_private_and_public_rows(private_rows, public_rows, public_ratio=PUBLIC_RATIO, seed=17)
mixed_manifest = RUN_ROOT / "mixed_train_manifest.jsonl"
kh.write_jsonl(mixed_manifest, mixed_rows)

public_count = sum(1 for row in mixed_rows if row.get("dataset_scope") == "public_asr")
print({"mixed_rows": len(mixed_rows), "public_rows_used": public_count, "public_ratio_actual": public_count / max(1, len(mixed_rows))})

In [ ]:
stage17 = REPO_ROOT / "stage_17_WER_acft_Whisper_Futo_finetuned_model_training_only_local_en_version_only_qat_dora.py"
stage17_cmd = [
    sys.executable,
    str(stage17),
    "--manifest-path", str(mixed_manifest),
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--futo-model-id", os.environ.get("FUTO_MODEL_ID", "futo-org/acft-whisper-small.en"),
    "--processor-id", os.environ.get("PROCESSOR_ID", "openai/whisper-small.en"),
    "--start-fresh", str(START_FRESH),
    "--set", "LR_START=1e-6",
    "--set", "MAX_EPOCHS=1",
    "--set", f"N_SAMPLES_PER_EPOCH={N_SAMPLES_PER_EPOCH}",
    "--set", "BATCH_SIZE=1",
    "--set", "GRAD_ACCUM_STEPS=1",
    "--set", "NUM_WORKERS=0",
    "--set", "DELETE_TRAINED_FROM_DRIVE=false",
]

result = kh.run_resumable_stage(
    "stage-17-train-smoke",
    stage17_cmd,
    inputs=[mixed_manifest],
    outputs=[CHECKPOINT_DIR],
    config={
        "profile": PROFILE,
        "lr_start": "1e-6",
        "max_epochs": 1,
        "n_samples_per_epoch": N_SAMPLES_PER_EPOCH,
        "start_fresh": START_FRESH,
    },
    state_dir=STATE_DIR,
    dry_run=DRY_RUN_STAGE17,
)
print(json.dumps(result, indent=2, sort_keys=True))

In [ ]:
handle = kh.make_dataset_handle(OWNER, "acft-kaggle-train", RUN_TAG, PROFILE)
kh.write_dataset_metadata(
    local_dir=RUN_ROOT,
    handle=handle,
    title=f"ACFT Kaggle train {RUN_TAG}",
    subtitle=f"{PROFILE} profile Stage 17 checkpoints and logs",
    keywords=["acft", "whisper", "training", "checkpoint", "resumable"],
    licenses=[{"name": "unknown"}],
)
publish = kh.publish_dataset(
    local_dir=RUN_ROOT,
    handle=handle,
    version_notes=f"{RUN_TAG}: stage-17-train-smoke",
    dry_run=DRY_RUN_PUBLISH,
)
print(json.dumps(publish, indent=2, sort_keys=True))
print("private eval manifest excludes public ASR rows:", private_eval_manifest)